# 06. Calibration — Feature Level

03/04와 동일한 surrogate 조건에서 Calibrated-Feature를 적용한다.

- Tree depth=1 (no mono / +mono) x 2 데이터셋
- EBM GAM (no mono / +mono) x 2 데이터셋
- Baseline 지표는 03/04와 동일해야 연속성 유지

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate, EBMSurrogate
from decentra.calibration import FeatureCalibrator

DATA_DIR = '../.data'

In [ ]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }

In [ ]:
def get_bb(data_flag):
    """BB ground truth. log-odds: contrib > 0 = 감점."""
    bb_contrib = base_models[data_flag]['bb_contribs']['lgb']
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    order = np.argsort(np.abs(bb_contrib), axis=1)[:, ::-1]
    bb_rank = feature_names[order]
    bb_adv = []
    for i in range(len(bb_contrib)):
        pos_idx = np.where(bb_contrib[i] > 0)[0]
        if len(pos_idx) == 0:
            bb_adv.append(np.array([], dtype='<U1'))
        else:
            pos_order = pos_idx[np.argsort(bb_contrib[i][pos_idx])[::-1]]
            bb_adv.append(feature_names[pos_order])
    return bb_contrib, bb_rank, bb_adv


def advfull_metrics(bb_adv_list, surr_adv_list):
    """AdvFull: Recall & Jaccard of adverse feature sets."""
    recalls, jaccards = [], []
    for i in range(len(bb_adv_list)):
        bb_set = set(bb_adv_list[i])
        su_set = set(surr_adv_list[i])
        if len(bb_set) == 0:
            continue
        inter = len(bb_set & su_set)
        recalls.append(inter / len(bb_set))
        union = len(bb_set | su_set)
        jaccards.append(inter / union if union > 0 else 0.0)
    r = round(float(np.mean(recalls)), 4) if recalls else 0.0
    j = round(float(np.mean(jaccards)), 4) if jaccards else 0.0
    return r, j


def advfull_from_contribs(contribs, bb_adv_list, feature_names):
    """AdvFull from raw contribs array."""
    surr_adv = []
    for i in range(len(contribs)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(neg_idx) == 0:
            surr_adv.append(np.array([], dtype='<U1'))
        else:
            neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
            surr_adv.append(feature_names[neg_order])
    return advfull_metrics(bb_adv_list, surr_adv)


def evaluate_surrogate(surr, X, y, bb_rank=None, bb_adv=None, k=4):
    """surrogate 객체 기반 평가. (FIXED: AdvFull에서 contributions 직접 계산)"""
    pred = surr.predict(X)
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    if bb_rank is not None and hasattr(surr, 'contribution_ranking'):
        surr_rank = surr.contribution_ranking(X)
        result[f'Top{k}'] = round(float(np.mean([
            len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(X))])), 4)
    if bb_adv is not None and hasattr(surr, 'adverse_features'):
        surr_adv_list = surr.adverse_features(X)
        adv_ov = []
        for i in range(len(X)):
            if len(bb_adv[i])==0 or len(surr_adv_list[i])==0: continue
            ke = min(k, len(bb_adv[i]), len(surr_adv_list[i]))
            adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv_list[i][:ke]))/ke)
        result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0

    # AdvFull: surrogate의 contributions와 feature_names를 직접 가져옴
    if bb_adv is not None and hasattr(surr, 'contributions'):
        surr_contribs = surr.contributions(X)
        fn = surr._feature_names(X) if hasattr(surr, '_feature_names') else np.array([f'f{i}' for i in range(surr_contribs.shape[1])])
        recall, jaccard = advfull_from_contribs(surr_contribs, bb_adv, fn)
        result['AdvFull_R'] = recall
        result['AdvFull_J'] = jaccard
    return result


def eval_contribs(contribs, pred, y, bb_rank, bb_adv, feature_names, k=4):
    """Calibrated contribs 평가."""
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    order = np.argsort(np.abs(contribs), axis=1)[:, ::-1]
    surr_rank = feature_names[order]
    result[f'Top{k}'] = round(float(np.mean([
        len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(y))])), 4)
    adv_ov = []
    for i in range(len(y)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(bb_adv[i])==0 or len(neg_idx)==0: continue
        neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
        surr_adv = feature_names[neg_order]
        ke = min(k, len(bb_adv[i]), len(surr_adv))
        adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv[:ke]))/ke)
    result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0

    recall, jaccard = advfull_from_contribs(contribs, bb_adv, feature_names)
    result['AdvFull_R'] = recall
    result['AdvFull_J'] = jaccard
    return result

In [ ]:
%%time
REJECT_PCTS = [5, 10, 20, 30, 40, 50]

for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val, y_test = t['y_logit_tr'], t['y_logit_val'], t['y_logit_test']
    bb_contrib, bb_rank, bb_adv = get_bb(data_flag)
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)

    # 원모형 확률 (test set)
    prob_test = base_models[data_flag]['lgb'].predict_proba(X_test)[:, 1]

    X_train = pd.concat([X_tr, X_val], ignore_index=True)
    y_train = np.concatenate([y_tr, y_val])

    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')

    surrogates = [
        ('Tree-d1',
         TreeSurrogate(max_depth=1, verbose=0),
         {'X': X_tr, 'y_logit': y_tr, 'eval_set': (X_val, y_val)}),
        ('Tree-d1+mono',
         TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0),
         {'X': X_tr, 'y_logit': y_tr, 'eval_set': (X_val, y_val)}),
        ('EBM-GAM',
         EBMSurrogate(interactions=0),
         {'X': X_train, 'y_logit': y_train}),
        ('EBM-GAM+mono',
         EBMSurrogate(interactions=0, monotone_detect_mode='auto'),
         {'X': X_train, 'y_logit': y_train}),
    ]

    for surr_name, surr_obj, fit_kwargs in surrogates:
        print(f'\n  === {surr_name} ===')
        surr_obj.fit(**fit_kwargs)
        surr_pred = surr_obj.predict(X_test)
        surr_contribs = surr_obj.contributions(X_test)

        # Baseline (전체)
        res_base = evaluate_surrogate(surr_obj, X_test, y_test, bb_rank, bb_adv)
        print(f'    Baseline (full): {res_base}')

        # Calibrated-Feature
        cal = FeatureCalibrator()
        cal_contribs, cal_pred = cal.fit_transform(surr_contribs, bb_contrib, surr_pred)
        res_cal = eval_contribs(cal_contribs, cal_pred, y_test, bb_rank, bb_adv, feature_names)
        print(f'    Cal-Feat (full):  {res_cal}')

        # 거절자 비율별 AdvTop4
        print(f'    --- reject subset AdvTop4 ---')
        for pct in REJECT_PCTS:
            # 원모형 확률 상위 pct% = 거절 대상
            cutoff = np.percentile(prob_test, 100 - pct)
            reject = prob_test >= cutoff
            if reject.sum() == 0:
                continue

            # Baseline AdvTop4 (거절자 한정)
            surr_adv_all = surr_obj.adverse_features(X_test)
            adv_base = []
            for i in range(len(X_test)):
                if not reject[i]: continue
                if len(bb_adv[i])==0 or len(surr_adv_all[i])==0: continue
                ke = min(4, len(bb_adv[i]), len(surr_adv_all[i]))
                adv_base.append(len(set(bb_adv[i][:ke]) & set(surr_adv_all[i][:ke]))/ke)
            at4_base = round(float(np.mean(adv_base)), 4) if adv_base else 0.0

            # Cal-Feature AdvTop4 (거절자 한정)
            adv_cal = []
            for i in range(len(X_test)):
                if not reject[i]: continue
                neg_idx = np.where(cal_contribs[i] < 0)[0]
                if len(bb_adv[i])==0 or len(neg_idx)==0: continue
                neg_order = neg_idx[np.argsort(cal_contribs[i][neg_idx])]
                surr_adv_i = feature_names[neg_order]
                ke = min(4, len(bb_adv[i]), len(surr_adv_i))
                adv_cal.append(len(set(bb_adv[i][:ke]) & set(surr_adv_i[:ke]))/ke)
            at4_cal = round(float(np.mean(adv_cal)), 4) if adv_cal else 0.0

            print(f'      reject={pct:>2d}%: base={at4_base}  cal={at4_cal}')